# Fine-tune the Nwoma NER model (Colab, free GPU)

Runtime > Change runtime type > **T4 GPU** first. Every step lives in `extract/train_ner.py`; this notebook only runs it.

1. Check the GPU, get the code, install pinned versions.
2. Upload `data/ner_train_dev.jsonl.gz` (train + dev only; the test set never leaves the laptop).
3. Rehearsal on 50 documents (~2 min): catches export or install errors before the real run.
4. Full run (~15-25 min), then download the int8 model in parts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Kobi-Ampem/nwoma.git
%cd nwoma
!git log --oneline -1
# Same Transformers version as the laptop: a version mismatch silently changes training arguments.
!pip install -q "transformers==5.20.2" accelerate sentencepiece onnx onnxruntime "pydantic>=2.8"

In [ ]:
from google.colab import files

uploaded = files.upload()  # choose data/ner_train_dev.jsonl.gz from the laptop

## Rehearsal: 50 documents, 1 epoch. The scores mean nothing; it only has to finish.

In [ ]:
!python -m extract.train_ner train --data ner_train_dev.jsonl.gz \
    --out ner_rehearsal --max-docs 50 --epochs 1

## Full run

In [ ]:
!python -m extract.train_ner train --data ner_train_dev.jsonl.gz --out ner_model

In [ ]:
# files.download() fails silently on large files, so send the model in 60 MB parts.
# On the laptop: cd ~/Downloads && cat ner_part_* > ner_model_int8.zip
import glob

from google.colab import files

!cd ner_model && zip -q ../ner_model_int8.zip model.int8.onnx train_report.json \
    tokenizer* special_tokens_map.json sentencepiece* 2>/dev/null
!split -b 60M -d ner_model_int8.zip ner_part_ && ls -lh ner_part_*
for part in sorted(glob.glob("ner_part_*")):
    files.download(part)